# Web Demo Steps

Notebook này gom các bước chuẩn bị, start web và test nhanh cho web demo recommendation + validation.

Chạy tuần tự từ trên xuống. Nếu đã có web server chạy ở `http://127.0.0.1:8001`, có thể bỏ qua cell start server.

## Step 1 - Kiểm tra thư mục project và file môi trường

Cell này xác nhận notebook đang chạy từ root project và kiểm tra các biến môi trường cần thiết. API key chỉ hiển thị trạng thái có/không, không in key ra màn hình.

In [ ]:
from pathlib import Path
from dotenv import dotenv_values

ROOT = Path.cwd()
if not (ROOT / "requirements.txt").exists() and (ROOT.parent / "requirements.txt").exists():
    ROOT = ROOT.parent

print("ROOT:", ROOT)
print("web/app.py:", (ROOT / "web" / "app.py").exists())
print("web/index.html:", (ROOT / "web" / "index.html").exists())
print("data:", (ROOT / "data" / "go_vap_tan_binh_100_enriched.json").exists())

env = dotenv_values(ROOT / ".env")
for key in ["GEOAPIFY_API_KEY", "MAPBOX_TOKEN", "OPENROUTER_API_KEYS", "SOLUTION1_DB_DSN"]:
    value = env.get(key) or ""
    print(f"{key}:", "configured" if value.strip() else "missing")

## Step 2 - Cài dependencies

Chạy cell này nếu môi trường Python chưa cài dependencies.

In [ ]:
# Uncomment nếu cần cài dependencies trong notebook environment.
# %pip install -r requirements.txt
# %pip install pytest

## Step 3 - Start Postgres cho Solution 1

Yêu cầu Docker Desktop đang chạy. Nếu chỉ demo `Solution 2`, bước này không bắt buộc.

In [ ]:
import subprocess

result = subprocess.run(
    ["docker", "compose", "up", "-d", "solution1_db"],
    cwd=ROOT,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
else:
    ps = subprocess.run(["docker", "compose", "ps"], cwd=ROOT, text=True, capture_output=True)
    print(ps.stdout)

## Step 4 - Start web server

Cell này start FastAPI ở port `8001` bằng entrypoint mới `web.app:app`. Nếu port đang bận, đổi biến `PORT`.

In [ ]:
import socket
import subprocess
import sys
import time

PORT = 8001

def port_is_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.5)
        return sock.connect_ex(("127.0.0.1", port)) == 0

if port_is_open(PORT):
    print(f"Server already running: http://127.0.0.1:{PORT}")
    WEB_PROCESS = None
else:
    WEB_PROCESS = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "web.app:app", "--port", str(PORT)],
        cwd=ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    time.sleep(3)
    print(f"Started server: http://127.0.0.1:{PORT}")
    print("PID:", WEB_PROCESS.pid)

## Step 5 - Smoke test API

Cell này kiểm tra frontend, validation cases, Solution 2 recommendation, validation hiện tại và batch validation.

In [ ]:
import requests

BASE_URL = f"http://127.0.0.1:{PORT}"

def show_response(name, response):
    print(name, response.status_code)
    response.raise_for_status()

show_response("GET /", requests.get(f"{BASE_URL}/", timeout=10))
cases_response = requests.get(f"{BASE_URL}/api/validation-cases", timeout=10)
show_response("GET /api/validation-cases", cases_response)
cases = cases_response.json()["cases"]
print("cases:", len(cases), "first:", cases[0]["id"])

form = {
    "budget_max_million": 8000,
    "min_bedrooms": 3,
    "soft_preferences": {
        "price": {"weight": 0.25, "direction": "lower_better", "min": 3000, "max": 8000},
        "distance_to_nearest_school_m": {"weight": 0.25, "direction": "lower_better", "min": 0, "max": 2000},
        "distance_to_nearest_park_m": {"weight": 0.2, "direction": "lower_better", "min": 0, "max": 2000},
        "distance_to_nearest_supermarket_m": {"weight": 0.15, "direction": "lower_better", "min": 0, "max": 1500},
        "area_m2": {"weight": 0.15, "direction": "higher_better", "min": 30, "max": 150},
    },
    "user_need_text": "Gia dinh co 2 con nho, uu tien gan truong va cong vien.",
}

recommend_payload = {
    "form": form,
    "free_text": form["user_need_text"],
    "solution": "solution2",
    "top_x": 5,
}
recommend_response = requests.post(f"{BASE_URL}/api/recommend", json=recommend_payload, timeout=30)
show_response("POST /api/recommend", recommend_response)
recommend_data = recommend_response.json()
s2 = recommend_data["results"]["solution2"]
print("solution2 status:", s2["status"])
print("top ids:", [item["property_id"] for item in s2["top5"]])

validate_payload = {
    **recommend_payload,
    "expected": {"soft_priorities": ["near school", "near park", "reasonable price"]},
}
validate_response = requests.post(f"{BASE_URL}/api/validate", json=validate_payload, timeout=30)
show_response("POST /api/validate", validate_response)
evaluation = validate_response.json()["evaluations"]["solution2"]
print("hard_pass:", evaluation["hard_constraints"]["pass_rate"])
print("priority_coverage:", evaluation["expected_priority_coverage"]["coverage"])

batch_response = requests.post(
    f"{BASE_URL}/api/validate-batch",
    json={"solution": "solution2", "dataset": "validation_cases_v1", "limit": 5, "top_x": 5},
    timeout=30,
)
show_response("POST /api/validate-batch", batch_response)
print("batch summary:", batch_response.json()["summary"])

## Optional - Test Solution 1 DB path không gọi LLM

Case này dùng hard constraints bất khả thi để xác nhận Postgres và dataset đã load được, nhưng không gọi LLM.

In [ ]:
s1_no_candidate_payload = {
    "form": {
        "budget_max_million": 1,
        "min_bedrooms": 99,
        "soft_preferences": {
            "price": {"weight": 1.0, "direction": "lower_better", "min": 0, "max": 1}
        },
        "user_need_text": "Case test DB: khong co nha nao phu hop.",
    },
    "free_text": "Case test DB: khong co nha nao phu hop.",
    "solution": "solution1",
    "top_x": 3,
}

response = requests.post(f"{BASE_URL}/api/recommend", json=s1_no_candidate_payload, timeout=30)
show_response("POST /api/recommend solution1 no_candidate", response)
s1 = response.json()["results"]["solution1"]
print("status:", s1["status"])
print("top5:", s1["top5"])
print("summary:", s1["explanation_summary"])

## Optional - Stop web server do notebook start

Chỉ chạy cell này nếu Step 4 đã tạo `WEB_PROCESS`. Nếu server đã chạy sẵn từ terminal, cell này không dừng server đó.

In [ ]:
if "WEB_PROCESS" in globals() and WEB_PROCESS is not None:
    WEB_PROCESS.terminate()
    WEB_PROCESS.wait(timeout=10)
    print("Stopped web server")
else:
    print("No notebook-started server to stop")